# CodeList Evaluation Module

To evaluate the quality of the generated codelists, in absence of clinicians to review our outputs, we decided to search for published papers that document the CodeList used to interrogate databases with their research questions. The question would be used as input query in our workflow, and the published codelist assumed to be the golden standard outcome, that our system should be able to retrieve given similar search terms.

This module compares codelists obtained at different stages of the workflow against the published reference codelist.

Confusion matrixes are plotted at each interrogation. Recall should be prioritized given the importance of getting an exhaustive list, and worse outcome of missing any relevant codes out.

# Packages

In [ ]:
import pandas as pd
import requests
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from google.colab import files
import json
from itertools import zip_longest
import re
import os
import zipfile
from urllib.parse import urlparse

pd.set_option('display.max_colwidth', None)

# Reference Codelist Import

## Import

This function imports any individual json file created from Evaluation TestSet Reformatting and saves it as a dataframe. It also extracts the Research Question, to pass as User Query

In [ ]:
def load_json_and_extract_query():
    # Upload file
    uploaded = files.upload()
    file_name = list(uploaded.keys())[0]

    # Load JSON
    with open(file_name, 'r') as f:
        data = json.load(f)

    # Convert to DataFrame
    df = pd.DataFrame(data)

    # Extract Research_question (assumes same value across rows)
    if "Research_question" in df.columns:
        user_query = df["Research_question"].dropna().iloc[0]
    else:
        user_query = None

    return df, user_query

In [ ]:
# Run the function
df_loaded, user_query = load_json_and_extract_query()

print(user_query)
df_loaded.head()

Saving Source_entry_9.json to Source_entry_9.json
ICD10 codes for a clinical diagnosis of delirium;


,Entry_no,Ref_link,Ref_reviewer,Ref_review_FLAG,Source_creation_date,Research_question,Codelist_vocabulary,Codelist,Codelist_terms,Terms_count,Codes_count,Diff_FLAG
0,9,https://datacompass.lshtm.ac.uk/id/eprint/4529/,AD,no - mock,1569715200000,ICD10 codes for a clinical diagnosis of delirium;,ICD-10,R26,Delirium,1,1,N


## Deduplication Step


In [ ]:
# Deduplicate Clinical_codes, concatenating Characteristics with "/" as a divider when multiple rows are merged
df_ref = df_loaded.groupby('Codelist')['Codelist_terms'].apply(lambda x: '/ '.join(x)).reset_index()
df_ref

,Codelist,Codelist_terms
0,R26,Delirium


# Import NICE Codelists

## Import jsons to dfs

In [ ]:
def load_jsons(github_api_url):
    """
    github_api_url example:
    https://api.github.com/repos/OWNER/REPO/contents/path/to/folder
    """

    skip = (
        "1_query_parser_output",
        "7b_ambiguous_codes",
        "8_final_output",
    )

    def filename_to_df(name):
        base = name.replace(".json", "")

        if base.startswith(skip):
            return None

        if base.startswith("2_retrieved_"):
            suffix = base[len("2_retrieved_"):]
            match = re.match(r"[A-Za-z0-9_]+", suffix)
            return "df_" + match.group(0) if match else None

        if base.startswith("6_merged_results"):
            return "df_merged"

        if base.startswith("7_scored_results"):
            return "df_scored"

        return None

    res = requests.get(github_api_url).json()

    dfs = {}

    for f in res:
        if not f["name"].endswith(".json"):
            continue

        df_name = filename_to_df(f["name"])
        if not df_name:
            continue

        data = requests.get(f["download_url"]).json()
        df = pd.json_normalize(data)

        dfs[df_name] = df
        globals()[df_name] = df  # optional: creates variables in notebook

        print(f"{f['name']} -> {df_name}")

    return dfs

In [ ]:
url = "https://api.github.com/repos/carlos-ramblox/nice-clinical-codes/contents/data/sample_outputs"
dfs = load_jsons(url)

2_retrieved_OMOPHub.json -> df_OMOPHub
2_retrieved_OpenCodelists_Bennett_Institute.json -> df_OpenCodelists_Bennett_Institute
2_retrieved_OpenCodelists_nhsd-primary-care-domain-r.json -> df_OpenCodelists_nhsd
2_retrieved_OpenCodelists_qcovid.json -> df_OpenCodelists_qcovid
2_retrieved_QOF_Business_Rules_2024-25.json -> df_QOF_Business_Rules_2024
6_merged_results.json -> df_merged
7_scored_results.json -> df_scored


## Summary table of imported files

In [ ]:
def unique_count_table(dfs):
    def make_hashable(x):
        if isinstance(x, list):
            return tuple(x)          # lists → tuple
        if isinstance(x, dict):
            return tuple(sorted(x.items()))  # dict → sorted tuple
        return x

    rows = []

    for df_name, df in dfs.items():
        row = {"df_name": df_name}

        for col in df.columns:
            try:
                series = df[col].apply(make_hashable)
                row[col] = series.nunique(dropna=True)
            except Exception:
                row[col] = None  # fallback if something weird happens

        rows.append(row)

    summary = pd.DataFrame(rows).set_index("df_name")
    return summary

In [ ]:
summary_table = unique_count_table(dfs)
summary_table

,code,term,vocabulary,source,domain,similarity_score,usage_frequency,sources,source_count,decision,confidence,rationale,classifier_score,llm_score
df_name,,,,,,,,,,,,,,
df_OMOPHub,50,50,2,1.0,3.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
df_OpenCodelists_Bennett_Institute,50,50,1,1.0,1.0,49.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
df_OpenCodelists_nhsd,50,49,1,1.0,1.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
df_OpenCodelists_qcovid,50,50,1,1.0,1.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
df_QOF_Business_Rules_2024,50,50,1,1.0,1.0,3.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
df_merged,100,100,1,3.0,1.0,85.0,0,10.0,2.0,NaN,NaN,NaN,NaN,NaN
df_scored,100,100,1,NaN,NaN,NaN,0,10.0,NaN,3.0,9.0,100.0,0.0,9.0


## Deduplication

In [ ]:
def deduplicate_rows(df):
    df = df.sort_values(by='similarity_score', ascending=False)

    # if duplicate codes exist
    if df['code'].nunique() != len(df):
        df = (
            df.groupby('code')
              .agg({
                  'term': lambda x: '/ '.join(dict.fromkeys(x)),  # deduplicate terms
                  'similarity_score': 'max'
              })
              .reset_index()
        )

    return df

In [ ]:
# Deduplicate all dfs
updated_dfs = {}
for df_name, df in dfs.items():
    if 'similarity_score' in df.columns:
        updated_dfs[df_name] = deduplicate_rows(df)
    else:
        # If 'similarity_score' is not present, perform basic deduplication if 'code' and 'term' exist
        if 'code' in df.columns and 'term' in df.columns and df['code'].nunique() != len(df):
            # Group by 'code' and concatenate unique 'term' values
            updated_dfs[df_name] = (
                df.groupby('code', as_index=False)
                  .agg(term=('term', lambda x: '/ '.join(dict.fromkeys(x))))
            )
        else:
            # If no 'similarity_score' and no 'code' or 'term' for basic deduplication,
            # or no duplicates by 'code', keep the original DataFrame
            updated_dfs[df_name] = df
dfs = updated_dfs

In [ ]:
summary_table_dedup = unique_count_table(dfs)
summary_table_dedup

,code,term,vocabulary,source,domain,similarity_score,usage_frequency,sources,source_count,decision,confidence,rationale,classifier_score,llm_score
df_name,,,,,,,,,,,,,,
df_OMOPHub,50,50,2,1.0,3.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
df_OpenCodelists_Bennett_Institute,50,50,1,1.0,1.0,49.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
df_OpenCodelists_nhsd,50,49,1,1.0,1.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
df_OpenCodelists_qcovid,50,50,1,1.0,1.0,0.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
df_QOF_Business_Rules_2024,50,50,1,1.0,1.0,3.0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
df_merged,100,100,1,3.0,1.0,85.0,0,10.0,2.0,NaN,NaN,NaN,NaN,NaN
df_scored,100,100,1,NaN,NaN,NaN,0,10.0,NaN,3.0,9.0,100.0,0.0,9.0


## Split scored based on decision

In [ ]:
# Save as individual df based on decision value
df_scored_include = df_scored[df_scored['decision'] == 'include']
df_scored_exclude = df_scored[df_scored['decision'] == 'exclude']
df_scored_review = df_scored[df_scored['decision'] == 'uncertain']

In [ ]:
# Add these to dfs
dfs['df_scored_include'] = df_scored_include
dfs['df_scored_exclude'] = df_scored_exclude
dfs['df_scored_review'] = df_scored_review

# Visualize list of dfs names in dfs
dfs.keys()

dict_keys(['df_OMOPHub', 'df_OpenCodelists_Bennett_Institute', 'df_OpenCodelists_nhsd', 'df_OpenCodelists_qcovid', 'df_QOF_Business_Rules_2024', 'df_merged', 'df_scored', 'df_scored_include', 'df_scored_exclude', 'df_scored_review'])

# Evaluation

## Functions definition

Function to evaluate a codelist against the reference. Returns the metrics table and the codelists for FP/FN/TP

In [ ]:
def codelist_evaluation(df_ref, df_output):

  # df_ref = gold standard codelist (published and peer-reviewed)
  # df_output = codelist from the workflow being evaluated

  # Create matrix that concatenates all codes across the dfs, and scores 0/1 based on the absence/presence of each code in each df
  all_codes_output = pd.concat([df_ref['Codelist'], df_output['code']]).unique()
  df_eval_output = pd.DataFrame(all_codes_output, columns=['code'])
  df_eval_output['ref_label'] = df_eval_output['code'].isin(df_ref['Codelist']).astype(int)
  df_eval_output['output_label'] = df_eval_output['code'].isin(df_output['code']).astype(int)

  # Count false positives (FP), false negatives (FN) and true positives (TP)
  FP_count = df_eval_output[(df_eval_output['ref_label'] == 0) & (df_eval_output['output_label'] == 1)].shape[0]
  FN_count = df_eval_output[(df_eval_output['ref_label'] == 1) & (df_eval_output['output_label'] == 0)].shape[0]
  TP_count = df_eval_output[(df_eval_output['ref_label'] == 1) & (df_eval_output['output_label'] == 1)].shape[0]

  # Capture metrics in df_eval_results
  df_eval_results = pd.DataFrame(columns=['Metric', 'Value'])
  df_eval_results = pd.concat([df_eval_results, pd.DataFrame([{'Metric': 'N Ref Codes', 'Value': len(df_ref)}])], ignore_index=True) # count of ref codes
  df_eval_results = pd.concat([df_eval_results, pd.DataFrame([{'Metric': 'N Output Codes', 'Value': len(df_output)}])], ignore_index=True) # count of codes in list
  df_eval_results = pd.concat([df_eval_results, pd.DataFrame([{'Metric': 'Recall', 'Value': recall_score(df_eval_output['ref_label'], df_eval_output['output_label'])}])], ignore_index=True) # recall
  df_eval_results = pd.concat([df_eval_results, pd.DataFrame([{'Metric': 'FN Count', 'Value': FN_count}])], ignore_index=True) # count of FN
  df_eval_results = pd.concat([df_eval_results, pd.DataFrame([{'Metric': 'FP Count', 'Value': FP_count}])], ignore_index=True) # count of FP
  df_eval_results = pd.concat([df_eval_results, pd.DataFrame([{'Metric': 'TP Count', 'Value': TP_count}])], ignore_index=True) # count of TP

  # Create mappings for code labels (if a code has multiple terms, join them into a single string divided by /)
  df_ref_map = df_ref.groupby('Codelist')['Codelist_terms'].apply(lambda x: '/ '.join(x)).to_dict()
  df_output_map = df_output.groupby('code')['term'].apply(lambda x: '/ '.join(x)).to_dict()

  # Save list of FN
  df_output_FN = df_eval_output[(df_eval_output['ref_label'] == 1) & (df_eval_output['output_label'] == 0)].copy()
  df_output_FN = df_output_FN.drop(columns=['ref_label', 'output_label'])
  df_output_FN['Codelist_terms'] = df_output_FN['code'].map(df_ref_map)

  # Save list of FP
  df_output_FP = df_eval_output[(df_eval_output['ref_label'] == 0) & (df_eval_output['output_label'] == 1)].copy()
  df_output_FP = df_output_FP.drop(columns=['ref_label', 'output_label'])
  df_output_FP['Codelist_terms'] = df_output_FP['code'].map(df_output_map)

  # Save list of TP
  df_output_TP = df_eval_output[(df_eval_output['ref_label'] == 1) & (df_eval_output['output_label'] == 1)].copy()
  df_output_TP = df_output_TP.drop(columns=['ref_label', 'output_label'])
  df_output_TP['Codelist_terms'] = df_output_TP['code'].map(df_ref_map)

  return df_eval_results, df_output_FN, df_output_FP, df_output_TP

Function to summarise metrics of all dfs

In [ ]:
def get_metrics(df_eval_results, name):

  metrics = {
      'Codelist': name,
      'N Ref Codes': df_eval_results[df_eval_results['Metric'] == 'N Ref Codes']['Value'].iloc[0],
      'N Output Codes': df_eval_results[df_eval_results['Metric'] == 'N Output Codes']['Value'].iloc[0],
      'Recall': df_eval_results[df_eval_results['Metric'] == 'Recall']['Value'].iloc[0],
      'FN Count': df_eval_results[df_eval_results['Metric'] == 'FN Count']['Value'].iloc[0],
      'FP Count': df_eval_results[df_eval_results['Metric'] == 'FP Count']['Value'].iloc[0],
      'TP Count': df_eval_results[df_eval_results['Metric'] == 'TP Count']['Value'].iloc[0]
  }
  return pd.DataFrame([metrics])

Function to investigate codelist to review from df_scored

In [ ]:
def uncertain_evaluation(df_ref, df_output):

  # df_ref = gold standard codelist (published and peer-reviewed)
  # df_output = codelist with codes classed as uncertain from LLM

  # Initialize df_eval as copy of df_output where only code / term / confidence / rationale features are kept
  df_eval_output = df_output[['code', 'term', 'confidence', 'rationale']].copy()

  # Add new column called "in_ref" and populate with "yes" if the code is present in df_ref and "no" otherwise
  df_eval_output['in_ref'] = df_eval_output['code'].isin(df_ref['Codelist']).astype(str)
  df_eval_output['in_ref'] = df_eval_output['in_ref'].replace({'True': 'yes', 'False': 'no'})

  # Sort by in_ref
  df_eval_output = df_eval_output.sort_values(by='in_ref')

  # Percentage of yes over total N rows
  yes_pct = (df_eval_output['in_ref'] == 'yes').sum()*100/df_eval_output.shape[0]

  return df_eval_output, yes_pct

Function to investigate excluded from df_scored

In [ ]:
def exclude_evaluation(df_ref, df_output):

  # df_ref = gold standard codelist (published and peer-reviewed)
  # df_output = codelist with codes classed as exclude from LLM

  # Initialize df_eval as copy of df_output where only code / term / confidence / rationale features are kept
  df_eval_output = df_output[['code', 'term', 'confidence', 'rationale']].copy()

  # Add new column called "in_ref" and populate with "yes" if the code is present in df_ref and "no" otherwise
  df_eval_output['in_ref'] = df_eval_output['code'].isin(df_ref['Codelist']).astype(str)
  df_eval_output['in_ref'] = df_eval_output['in_ref'].replace({'True': 'yes', 'False': 'no'})

  # Sort by in_ref
  df_eval_output = df_eval_output.sort_values(by='in_ref')

  # Count yes
  yes_count = (df_eval_output['in_ref'] == 'yes').sum()

  # Count yes over total N rows
  yes_pct = (df_eval_output['in_ref'] == 'yes').sum()*100/df_eval_output.shape[0]

  return df_eval_output, yes_pct, yes_count

Function to run codelist_evaluation to all dfs except df_scored (since this df has been divided in included/excluded/review)

In [ ]:
def save_eval_runs(df_ref, dfs, out_dir="eval_outputs"):
    os.makedirs(out_dir, exist_ok=True)

    exclude = {"df_scored", "df_scored_exclude", "df_scored_review"}
    metrics_list = []  # collect metrics here

    for name, df in dfs.items():
        if name in exclude:
            continue

        df_results, df_FN, df_FP, df_TP = codelist_evaluation(df_ref, df)

        # save files
        #df_results.to_json(f"{out_dir}/{name}_results.json", orient="records", indent=2)
        df_FN.to_json(f"{out_dir}/{name}_FN.json", orient="records", indent=2)
        df_FP.to_json(f"{out_dir}/{name}_FP.json", orient="records", indent=2)
        df_TP.to_json(f"{out_dir}/{name}_TP.json", orient="records", indent=2)

        # collect metrics
        metrics_df = get_metrics(df_results, name)  # your function
        metrics_list.append(metrics_df)

        # handle exclude and review lists
        # extra: df_scored_exclude
    if "df_scored_exclude" in dfs:
        df_eval_exclude_scored, yes_pct_exclude_scored, yes_count_exclude_scored = exclude_evaluation(
            df_ref, dfs["df_scored_exclude"]
        )

        df_eval_exclude_scored.to_json(
            f"{out_dir}/df_scored_exclude_results.json", orient="records", indent=2
        )

        pd.DataFrame([{
            "df_name": "df_scored_exclude",
            "yes_pct_exclude_scored": yes_pct_exclude_scored,
            "yes_count_exclude_scored": yes_count_exclude_scored,
        }]).to_json(
            f"{out_dir}/df_scored_exclude_summary.json", orient="records", indent=2
        )

    # extra: df_scored_review
    if "df_scored_review" in dfs:
        df_eval_uncertain_scored, yes_pct_uncertain_scored = uncertain_evaluation(
            df_ref, dfs["df_scored_review"]
        )

        df_eval_uncertain_scored.to_json(
            f"{out_dir}/df_scored_review_results.json", orient="records", indent=2
        )

        pd.DataFrame([{
            "df_name": "df_scored_review",
            "yes_pct_uncertain_scored": yes_pct_uncertain_scored,
        }]).to_json(
            f"{out_dir}/df_scored_review_summary.json", orient="records", indent=2
        )

        # compile metrics summary
        df_metrics_summary = pd.concat(metrics_list, ignore_index=True)
        df_metrics_summary.to_json(f"{out_dir}/df_metrics_summary.json", orient="records", indent=2)

    # zip everything
    zip_path = f"{out_dir}/df_ref_eval_outputs.zip"
    with zipfile.ZipFile(zip_path, "w") as z:
        for f in os.listdir(out_dir):
            if f.endswith(".json"):
                z.write(os.path.join(out_dir, f), f)

    return zip_path

## Results on codelists

Call codelist_evaluation and get results for all dfs except df_scored, df_scored_review, df_scored_exclude

In [ ]:
# Run on set of dfs and save in zip
zip_path = save_eval_runs(df_ref, dfs)
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Visualisation of summary

In [ ]:
df_metrics_summary = pd.read_json("eval_outputs/df_metrics_summary.json")
df_metrics_summary

,Codelist,N Ref Codes,N Output Codes,Recall,FN Count,FP Count,TP Count
0,df_OMOPHub,1,50,0,1,50,0
1,df_OpenCodelists_Bennett_Institute,1,50,0,1,50,0
2,df_OpenCodelists_nhsd,1,50,0,1,50,0
3,df_OpenCodelists_qcovid,1,50,0,1,50,0
4,df_QOF_Business_Rules_2024,1,50,0,1,50,0
5,df_merged,1,100,0,1,100,0
6,df_scored_include,1,55,0,1,55,0


In [ ]:
df_exclude_summary = pd.read_json("eval_outputs/df_scored_exclude_summary.json")
yes_pct_exclude_scored = df_exclude_summary['yes_pct_exclude_scored'].iloc[0]
yes_count_exclude_scored = df_exclude_summary['yes_count_exclude_scored'].iloc[0]

print(yes_pct_exclude_scored, "% of the codes automatically excluded were actually present in the reference list (", yes_count_exclude_scored, " codes)")

0 % of the codes automatically excluded were actually present in the reference list ( 0  codes)


In [ ]:
df_review_summary = pd.read_json("eval_outputs/df_scored_review_summary.json")
yes_pct_uncertain_scored = df_review_summary['yes_pct_uncertain_scored'].iloc[0]
print(yes_pct_uncertain_scored, "% of the codes to review was included in the reference list")

0 % of the codes to review was included in the reference list
